# Program 14: Structured output with Pydantic and a custom parser

**Aim.** Take an institution name, define the output schema with Pydantic, write a custom output parser, invoke the chain and fetch details from Wikipedia.

**Theory.** Pydantic validates types and rules. A custom `BaseOutputParser` turns model text into a validated object and raises a clear error otherwise. The model may use only the Wikipedia text it is given, and must say `not stated` for anything absent, so it cannot invent facts.

In [1]:
import sys
sys.path.insert(0, '.')
from common import *


In [2]:
import json, re, wikipedia
from pydantic import BaseModel, Field, field_validator
from langchain_core.output_parsers import BaseOutputParser

class Institution(BaseModel):
    founder: str = Field(description='who founded it')
    founded: str = Field(description='when')
    branches: list[str] = Field(description='current schools, branches or campuses')
    employees: str = Field(description='number of employees')
    summary: list[str] = Field(description='exactly 4 short lines')

    @field_validator('summary')
    @classmethod
    def four_lines(cls, v):
        if len(v) != 4:
            raise ValueError(f'summary must have 4 lines, got {len(v)}')
        return v

class InstitutionParser(BaseOutputParser[Institution]):
    def parse(self, text: str) -> Institution:
        m = re.search(r'\{.*\}', text, re.S)
        if not m:
            raise ValueError('no JSON object in the model output')
        return Institution.model_validate(json.loads(m.group()))
    @property
    def _type(self): return 'institution'
print(list(Institution.model_fields))

['founder', 'founded', 'branches', 'employees', 'summary']


In [3]:
name = 'RV University'
hits = wikipedia.search(name)
print('wikipedia search:', hits[:4])
page = wikipedia.page(hits[0], auto_suggest=False)
source = page.content[:7000]
print('page:', page.title, '-', len(source.split()), 'words used')

wikipedia search: ['RV University', 'Rashtreeya Sikshana Samithi Trust', 'Pattanagere metro station', 'RV Knorr']


page: RV University - 1012 words used


In [4]:
from langchain_core.prompts import PromptTemplate
llm, label = get_llm()
print('model:', label)
prompt = PromptTemplate.from_template(
  'Using ONLY the text below, return JSON with keys founder, founded, branches (list), employees, summary (a list of EXACTLY 4 short lines) for {name}. '
  'If the text does not say, write "not stated". Do not use outside knowledge.\n\nTEXT:\n{text}')
chain = prompt | llm | InstitutionParser()
for attempt in range(2):
    try:
        result = chain.invoke({'name': name, 'text': source}); break
    except Exception as e:
        print('retry after:', e)
print(result.model_dump_json(indent=1))

model: Groq openai/gpt-oss-20b (no Cohere key available)


{
 "founder": "Rashtreeya Sikshana Samithi Trust (RSST)",
 "founded": "2021",
 "branches": [
  "Bengaluru",
  "Mysuru"
 ],
 "employees": "not stated",
 "summary": [
  "Founded in 2021 by RSST.",
  "Campus in Bengaluru and Mysuru.",
  "Offers UG, PG, PhD across 18 schools.",
  "Collaborates with global universities."
 ]
}


## Observations

Typing and the 4-line rule are enforced by code, not trusted to the prompt. "not stated" is an honest answer; it is better than a guess.